# Steadyhand: agent-loop Colab notebook

This notebook runs Steadyhand, the agent loop in `src/steadyhand/`, against a self-hosted model served through the Toolsmith shim. Steadyhand validates each native tool call and runs it against an in-memory mock store. On a tool failure it applies a recovery policy, and it writes one journal per episode. Handlers run in Steadyhand after `chat()` returns. The shim never executes a tool. The Toolsmith track itself lives in `notebooks/01_toolsmith.ipynb`.

Run the cells top to bottom on a fresh Colab T4 or L4 runtime:

1. **Setup.** Pulls the repository, mounts Drive, loads `HF_TOKEN` and `LOCAL_KEY`, installs the project, and detects the GPU, vCPUs, and RAM that size everything below.
2. **Runtime.** Starts restoring or building llama.cpp and copying the GGUF for the model in `STEADYHAND_PHASE`, in the background. It uses the same Drive caches as the Toolsmith notebook, through `toolsmith.provision`.
3. **CPU gate.** Runs the dataset validator, all six recovery policies on a scripted model, and the golden replays, on every vCPU, while the runtime provisions. It needs no GPU and must pass first.
4. **Server, checks, sweep.** Starts one llama-server for the phase's shape, checks that episodes overlap and that every slot is busy, and runs the sweep. Journals are copied to `MyDrive/toolsmith/records/<gpu>/steadyhand/` (`t4` or `l4`) and `RESULTS.md` is rendered from them.
5. **Stop** before the next phase or model, and **cleanup** at the end.

One GGUF is resident at a time. To switch models, set `STEADYHAND_PHASE` (`qwen35/wide`, `qwen35/tall`, `qwen3/wide`, `qwen3/tall`), then rerun the runtime cell and everything after it. Put `HF_TOKEN` and `LOCAL_KEY` in `MyDrive/toolsmith/.env` (or `.env.txt`) or in Colab Secrets, never in a cell. Record Colab units before and after the session.

**After a disconnect or a hung session:** reconnect, then run all cells from the top. Builds and models come back from Drive; the CPU gate reruns in minutes, and the sweep restores its scores from Drive and skips finished episodes; only the work that was in flight (and, at worst, the last minute or two not yet copied to Drive) is redone. If the server dies mid-sweep it is restarted once, and episodes it aborted are rerun. Keep the browser tab open during long cells: on Colab Pro (unlike Pro+) a session generally does not keep running in the background.

In [ ]:
# Standard-library imports used throughout, and a first look at the GPU and Python version.
# Runs before anything else; it must work even on a CPU runtime.
import json
import os
import platform
import shlex
import shutil
import socket
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

# No check=True: on a CPU runtime the hardware cell explains how to switch instead.
gpu_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True,
    text=True,
) if shutil.which('nvidia-smi') else None
gpu_info = gpu_result.stdout.strip() if gpu_result is not None and gpu_result.returncode == 0 else 'No NVIDIA GPU visible'
print(gpu_info)
print(f'Python {platform.python_version()}')

In [ ]:
# Clone or update the repository, mount Google Drive, create the shared cache folders, and load
# HF_TOKEN and LOCAL_KEY from MyDrive/toolsmith/.env or Colab Secrets (never from a cell).
from google.colab import drive, userdata

REPO_URL = 'https://github.com/shreyasnarahari/bare-metal-agent'
PROJECT_DIR = Path('/content/bare-metal-agent')
if (PROJECT_DIR / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', 'main'],
        check=True,
    )
elif PROJECT_DIR.exists():
    raise FileNotFoundError(
        f'{PROJECT_DIR} exists but is not a bare-metal-agent git checkout.'
    )
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
if not (PROJECT_DIR / 'pyproject.toml').is_file():
    raise FileNotFoundError(f'Expected {PROJECT_DIR / "pyproject.toml"} after repository setup.')
src_dir = PROJECT_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

drive.mount('/content/drive')
from toolsmith.provision import read_env_file

DRIVE_ROOT = Path('/content/drive/MyDrive/toolsmith')
DRIVE_BUILD_CACHE = DRIVE_ROOT / 'builds'
DRIVE_MODEL_CACHE = DRIVE_ROOT / 'models'
# Builds (keyed by compute capability) and models are shared by every GPU; records are per GPU.
for directory in (DRIVE_BUILD_CACHE, DRIVE_MODEL_CACHE):
    directory.mkdir(parents=True, exist_ok=True)

env_path = next(
    (
        candidate
        for candidate in (DRIVE_ROOT / '.env', DRIVE_ROOT / '.env.txt')
        if candidate.is_file()
    ),
    DRIVE_ROOT / '.env',
)
env_values = read_env_file(env_path)
def load_secret(name):
    value = env_values.get(name, '').strip()
    if value:
        return value, env_path
    return userdata.get(name), 'Colab Secrets'

HF_TOKEN, hf_source = load_secret('HF_TOKEN')
LOCAL_KEY, local_source = load_secret('LOCAL_KEY')
if not HF_TOKEN or not LOCAL_KEY:
    raise ValueError(f'HF_TOKEN and LOCAL_KEY must be set in {env_path} or Colab Secrets.')
print(f'HF_TOKEN loaded from {hf_source}; LOCAL_KEY loaded from {local_source}')

In [ ]:
# Install this repository (editable, with the Colab extras) into the notebook's Python.
get_ipython().run_line_magic(
    'pip',
    f"install -q -e {shlex.quote(str(PROJECT_DIR) + '[colab]')}",
)

In [ ]:
# Detect the GPU, vCPUs, RAM, and disk once; every later size (build jobs, slots, pools) comes from
# HARDWARE. Also sets the per-GPU records folder, the background-job runner, and the record stamp.
import os
import uuid
from functools import partial
from pathlib import Path

from toolsmith.background import BackgroundJobs
from toolsmith.drive import ensure_drive, sync_drive
from toolsmith.hardware import detect_hardware, require_supported, runtime_stamp

# Every build, server, and pool size below comes from what this runtime has (T4, L4, or other).
HARDWARE = detect_hardware()
print(HARDWARE.summary())
require_supported(HARDWARE)
GPU_FAMILY = HARDWARE.family
CPU_COUNT = HARDWARE.cpu_count
# A T4 run and an L4 run never share a checkpoint, report, or JSONL file.
DRIVE_RECORDS = DRIVE_ROOT / 'records' / HARDWARE.records_slug
DRIVE_RECORDS.mkdir(parents=True, exist_ok=True)
print(f'Records for this GPU: {DRIVE_RECORDS}')
# Network calls give up instead of hanging: git aborts a transfer under 1 KB/s for a minute, and
# Hugging Face requests time out (read when huggingface_hub is first imported, after this cell).
for name, value in {
    'GIT_HTTP_LOW_SPEED_LIMIT': '1000', 'GIT_HTTP_LOW_SPEED_TIME': '60',
    'HF_HUB_DOWNLOAD_TIMEOUT': '60', 'HF_HUB_ETAG_TIMEOUT': '30',
}.items():
    os.environ.setdefault(name, value)
# Downloads, builds, and Drive copies run here while the GPU works; each job logs to its own file.
if 'BACKGROUND' not in globals():
    BACKGROUND = BackgroundJobs(Path(os.environ.get('TOOLSMITH_BACKGROUND_LOGS', '/content/background-logs')))
# Added to every Toolsmith record: the GPU, how busy the machine was, and this kernel's session, so
# records rewritten by a rerun after a disconnect can be told apart from the first run's.
if 'SESSION_ID' not in globals():
    SESSION_ID = uuid.uuid4().hex[:12]
RUNTIME_STAMP = partial(runtime_stamp, HARDWARE, BACKGROUND, session_id=SESSION_ID)

## Runtime for the selected model

Each profile pins the llama.cpp tag and GGUF that the Toolsmith notebook verified for it: Qwen3-8B Q4_K_M on llama.cpp `b5401`, and the Qwen3.5-9B Q4_K_M language file on `b8233`. Builds and weights come from the shared Drive cache when present. Every executable is checked with `--version`, and every GGUF by magic bytes and exact size.

This cell only starts the work as background jobs and returns. The CPU gate below runs while the weights download or copy from Drive and llama.cpp restores or builds; the server cell waits for whatever is left. Build jobs follow the runtime's vCPUs and RAM, so a T4 compiles with two jobs instead of running out of memory.

In [ ]:
# Start provisioning the model STEADYHAND_PHASE names (llama.cpp build, GGUF, renderer) as
# background jobs and return at once; the server cell collects them after the CPU gate.
from toolsmith.prompting import ChatPromptRenderer
from toolsmith.protocols import QWEN3_PROFILE, QWEN35_PROFILE
from toolsmith.provision import (
    LlamaCppRuntime,
    cache_dir_for,
    ensure_gguf,
    ensure_llama_server,
    q4_k_m_language_file,
    resolve_gguf,
)

STEADYHAND_MODELS = {
    'qwen3': {'profile': QWEN3_PROFILE, 'llama_cpp_ref': 'b5401',
              'repo': 'Qwen/Qwen3-8B-GGUF', 'filename': 'Qwen3-8B-Q4_K_M.gguf'},
    'qwen35': {'profile': QWEN35_PROFILE, 'llama_cpp_ref': 'b8233',
               'repo': 'unsloth/Qwen3.5-9B-GGUF', 'filename': None},
}
steadyhand_profile_name = os.environ.get('STEADYHAND_PHASE', 'qwen35/wide').split('/')[0]
if steadyhand_profile_name not in STEADYHAND_MODELS:
    raise ValueError(f'STEADYHAND_PHASE must start with one of {sorted(STEADYHAND_MODELS)}')
steadyhand_model_spec = STEADYHAND_MODELS[steadyhand_profile_name]
if globals().get('STEADYHAND_ACTIVE'):
    raise RuntimeError('A Steadyhand llama-server is running; run the stop cell before switching models')

GPU_NAME = HARDWARE.gpu_name
GPU_COMPUTE_CAP = HARDWARE.compute_cap
steadyhand_ref = steadyhand_model_spec['llama_cpp_ref']
steadyhand_llama_runtime = LlamaCppRuntime(
    ref=steadyhand_ref,
    source=Path(f'/content/llama.cpp-{steadyhand_ref}-source'),
    build=Path(f'/content/llama.cpp-{steadyhand_ref}-build'),
    cache=cache_dir_for(DRIVE_BUILD_CACHE, steadyhand_ref, GPU_COMPUTE_CAP),
    compute_cap=GPU_COMPUTE_CAP,
    staging=Path(f'/content/llama.cpp-{steadyhand_ref}-staging'),
)

def provision_steadyhand_gguf(spec, profile_name):
    if spec['filename']:
        gguf = resolve_gguf(spec['repo'], token=HF_TOKEN, filename=spec['filename'])
        drive_name = gguf.filename
    else:
        gguf = resolve_gguf(spec['repo'], token=HF_TOKEN, select=q4_k_m_language_file)
        drive_name = f'{gguf.revision[:12]}-{Path(gguf.filename).name}'
    # The same Drive names the Toolsmith notebook uses, so both notebooks share one cached copy.
    local = ensure_gguf(
        gguf,
        local_path=Path('/content') / f'{profile_name}-{Path(gguf.filename).name}',
        drive_path=DRIVE_MODEL_CACHE / drive_name,
        token=HF_TOKEN,
        cache_dir=f'/content/gguf-download-{profile_name}',
        defer=BACKGROUND.submit,
    )
    return gguf, local

# Started here and collected by the server cell, so the CPU gate below runs while the weights
# download or copy from Drive and llama.cpp restores or builds.
BACKGROUND.submit(
    f'steadyhand-llama.cpp-{steadyhand_ref}', ensure_llama_server, steadyhand_llama_runtime,
    jobs=HARDWARE.build_jobs(), defer=BACKGROUND.submit,
)
BACKGROUND.submit(f'steadyhand-gguf-{steadyhand_profile_name}', provision_steadyhand_gguf, steadyhand_model_spec, steadyhand_profile_name)
BACKGROUND.submit(
    f'steadyhand-renderer-{steadyhand_profile_name}', ChatPromptRenderer.from_pretrained,
    steadyhand_model_spec['profile'].tokenizer, token=HF_TOKEN,
)
STEADYHAND_PROVISIONING = {
    'profile_name': steadyhand_profile_name,
    'profile': steadyhand_model_spec['profile'],
    'llama_cpp_ref': steadyhand_ref,
    'build': steadyhand_llama_runtime.build,
    'gpu': GPU_NAME,
    'compute_cap': GPU_COMPUTE_CAP,
}
STEADYHAND_RUNTIME = None

def collect_steadyhand_runtime():
    provisioning = STEADYHAND_PROVISIONING
    name, ref = provisioning['profile_name'], provisioning['llama_cpp_ref']
    server_bin = BACKGROUND.wait(f'steadyhand-llama.cpp-{ref}')
    gguf, local_model = BACKGROUND.wait(f'steadyhand-gguf-{name}')
    return {
        **provisioning,
        'server_bin': server_bin,
        'model': local_model,
        'model_repo': gguf.repo,
        'model_file': gguf.filename,
        'model_revision': gguf.revision,
        'model_size': gguf.size,
        'renderer': BACKGROUND.wait(f'steadyhand-renderer-{name}'),
    }

print(f'Provisioning {steadyhand_profile_name} on llama.cpp {steadyhand_ref} in the background: {BACKGROUND.status()}')

## Evaluation sweep (part 7)

The loop is scored on 60 mock-store tasks (`evaluation/data/steadyhand_tasks_v1.jsonl`). Each task's gold is a diff of the final store, not the wording of the answer. Each task runs clean and under every fault variant it declares. The fault seed is the task seed, and policies share the same pinned faults.

The server shape comes from `STEADYHAND_PHASE` and the GPU (a card other than a T4 or L4 borrows the rows for its VRAM). Context and slots are set as a pair: `per_slot_context = context_size // parallel`. The server cell tries a q8_0 KV cache with flash attention, then f16 with it, then f16 without it, and keeps the first that loads, leaves 1 GB of VRAM free, and passes a decode-speed self-test; running out of VRAM drops one slot at the same per-slot context. The smoke cell accepts the shape only when every slot is busy. The sweep runs `workers == parallel`, mirrors its scores to Drive as it goes, and resumes from them on a fresh runtime, even if the slot count changed; a different GPU or KV setting stops it.

Choose the grid with `STEADYHAND_POLICIES` (default `typed_hint,escalate`, the first live table), `STEADYHAND_PROMPTS` (`plain`, optionally `failure_aware`), and `STEADYHAND_VARIANTS`. The presets are capacity guesses for the probe to confirm, not measurements. Infrastructure aborts stay in the log and never count as task results.

In [ ]:
# CPU gate: validate the 60 tasks, run every recovery policy on the scripted model, and replay the
# golden journals. Runs while the runtime provisions, and must pass before any GPU episode.
import json
import os
import tempfile
from pathlib import Path

from steadyhand.eval.suite import load_tasks, run_cpu_suite, validate_tasks
from steadyhand.recovery import STRATEGIES
from steadyhand.replay import replay_many
from toolsmith.recording import JsonlRecorder

STEADYHAND_GOLDEN_DIR = Path(PROJECT_DIR) / 'tests' / 'fixtures' / 'steadyhand_episodes'
steadyhand_tasks = load_tasks()
steadyhand_dataset = validate_tasks(steadyhand_tasks)
steadyhand_gate_limit = int(os.environ.get('STEADYHAND_CPU_GATE_TASKS') or 0)
steadyhand_gate_tasks = steadyhand_tasks[:steadyhand_gate_limit] if steadyhand_gate_limit else steadyhand_tasks
steadyhand_gate_workers = int(os.environ.get('STEADYHAND_CPU_GATE_WORKERS') or HARDWARE.cpu_workers())
steadyhand_gate_dir = Path(tempfile.mkdtemp(prefix='steadyhand-cpu-gate-'))

# Every policy on the scripted model, then a blind-retry pass on the lost-ack slice.
steadyhand_cpu_scores = run_cpu_suite(
    steadyhand_gate_tasks, sorted(STRATEGIES),
    directory=steadyhand_gate_dir / 'follow-hints', workers=steadyhand_gate_workers,
)
steadyhand_blind_scores = run_cpu_suite(
    steadyhand_gate_tasks, ['raw_error', 'retry_safe', 'idempotent_verify', 'escalate'],
    directory=steadyhand_gate_dir / 'blind-retry', behavior='blind_retry',
    variants=['lost_acknowledgment'], workers=steadyhand_gate_workers,
)
steadyhand_golden = replay_many(sorted(STEADYHAND_GOLDEN_DIR.glob('ep_*.jsonl')), workers=steadyhand_gate_workers)

steadyhand_gate_problems = [
    f'{score.task_id} clean failed under {score.policy}'
    for score in steadyhand_cpu_scores
    if score.variant == 'clean' and score.policy in ('typed_hint', 'escalate') and not score.task_success
]
steadyhand_gate_problems += [
    f'{score.task_id} duplicated an effect under {score.policy}'
    for score in steadyhand_blind_scores
    if score.policy in ('idempotent_verify', 'escalate') and score.duplicate_side_effects
]
steadyhand_gate_problems += [f'golden replay failed: {outcome.error}' for outcome in steadyhand_golden if not outcome.ok]
steadyhand_gate_record = {
    'event': 'steadyhand_cpu_gate',
    'dataset': steadyhand_dataset,
    'tasks_run': len(steadyhand_gate_tasks),
    'episodes': len(steadyhand_cpu_scores) + len(steadyhand_blind_scores),
    'golden_replays': len(steadyhand_golden),
    'problems': steadyhand_gate_problems,
    'passed': not steadyhand_gate_problems,
}
JsonlRecorder(
    Path(DRIVE_RECORDS) / 'steadyhand' / 'cpu-gate.jsonl', secrets=(HF_TOKEN, LOCAL_KEY),
).append(steadyhand_gate_record)
print(json.dumps({key: value for key, value in steadyhand_gate_record.items() if key != 'dataset'}, indent=2))
if steadyhand_gate_problems:
    raise RuntimeError('Steadyhand CPU gate failed; fix it before any scored GPU episode')

In [ ]:
# Collect the provisioned runtime and start llama-server in the best shape this GPU holds (KV cache
# and flash-attention ladder, slots shrunk on OOM, decode self-test). Produces STEADYHAND_ACTIVE.
import subprocess
import time

from steadyhand import LiveSettings, build_llama_client
from steadyhand.eval.serving import (
    SELF_TEST_PROMPT,
    decode_speed,
    fit_server,
    kv_ladder,
    parse_nvidia_smi,
    preset_for,
    server_extra_args,
)
from toolsmith.backends import CompletionTruncatedError, LlamaCppClient, LlamaCppServer
from toolsmith.provision import free_port, server_environment

def stop_steadyhand_runtime():
    global STEADYHAND_ACTIVE
    active = globals().get('STEADYHAND_ACTIVE') or {}
    if active.get('client') is not None:
        active['client'].close()
    if active.get('server') is not None:
        active['server'].stop(timeout=30)
    STEADYHAND_ACTIVE = {}
    os.environ.pop('TOOLSMITH_BASE_URL', None)

def query_gpu():
    output = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.used,utilization.gpu',
         '--format=csv,noheader,nounits'],
        check=True, capture_output=True, text=True,
    ).stdout
    return parse_nvidia_smi(output)[0]

if not globals().get('STEADYHAND_PROVISIONING'):
    raise RuntimeError('Run the Steadyhand runtime cell first')
steadyhand_profile_name, steadyhand_shape = os.environ.get('STEADYHAND_PHASE', 'qwen35/wide').split('/')
if steadyhand_profile_name != STEADYHAND_PROVISIONING['profile_name']:
    raise RuntimeError(
        f"STEADYHAND_PHASE asks for {steadyhand_profile_name}, but the runtime cell provisioned "
        f"{STEADYHAND_PROVISIONING['profile_name']}; stop the server and rerun the runtime cell"
    )
# Waits for whatever the runtime cell's background jobs have not finished during the CPU gate.
if not globals().get('STEADYHAND_RUNTIME'):
    STEADYHAND_RUNTIME = collect_steadyhand_runtime()
steadyhand_runtime = STEADYHAND_RUNTIME

steadyhand_log = Path('/content/steadyhand-server.log')

def start_steadyhand_server(preset, extra_args, port):
    server = LlamaCppServer(
        executable=steadyhand_runtime['server_bin'], model=steadyhand_runtime['model'], api_key=LOCAL_KEY,
        log_path=steadyhand_log, port=port, context_size=preset.context_size,
        gpu_layers=99, parallel=preset.parallel, extra_args=extra_args,
    )
    probe = LlamaCppClient(server.base_url, LOCAL_KEY, timeout=30.0)
    try:
        server.start()
        server.wait_until_healthy(probe, timeout=900, poll_interval=2)
    except BaseException:
        server.stop(timeout=30)
        raise
    finally:
        probe.close()
    return server

def probe_decode_speed(server):
    # One short completion: a GPU-resident model decodes far above the floor, a CPU fallback does not.
    with LlamaCppClient(server.base_url, LOCAL_KEY, timeout=300.0) as client:
        started = time.perf_counter()
        try:
            response = client.complete(SELF_TEST_PROMPT, max_tokens=64, temperature=0.0, seed=0)
        except CompletionTruncatedError as error:
            response = error.response
        return decode_speed(response, time.perf_counter() - started)

def start_steadyhand_runtime():
    # Stops any Steadyhand server, then fits and starts this phase's. The sweep cell calls it
    # again if the server dies mid-sweep.
    global STEADYHAND_ACTIVE
    # One resident GGUF: a previous Steadyhand server is stopped before this one loads.
    stop_steadyhand_runtime()
    steadyhand_env = server_environment(steadyhand_runtime['build'])
    os.environ['LD_LIBRARY_PATH'] = steadyhand_env['LD_LIBRARY_PATH']
    steadyhand_help = subprocess.run(
        [str(steadyhand_runtime['server_bin']), '--help'], capture_output=True, text=True, env=steadyhand_env,
    ).stdout
    steadyhand_gpu = query_gpu()
    # A card that is neither a T4 nor an L4 borrows the preset rows for its VRAM.
    steadyhand_preset = preset_for(
        steadyhand_gpu['name'], steadyhand_profile_name, steadyhand_shape, vram_mib=steadyhand_gpu['memory_total_mib'],
    )
    steadyhand_port = free_port('127.0.0.1', 8090)
    # Tries q8_0 KV with flash attention, then f16 with it, then f16 without it. Running out of VRAM
    # costs one slot (same per-slot context); a crash or a slow self-test moves to the next setting.
    steadyhand_fit = fit_server(
        steadyhand_preset, kv_ladder(steadyhand_help),
        start=lambda preset, extra_args: start_steadyhand_server(preset, extra_args, steadyhand_port),
        extra_args_for=lambda rung: server_extra_args(
            steadyhand_help, cpu_count=HARDWARE.cpu_count, kv_cache=rung['kv_cache'], flash_attn=rung['flash_attn'],
        ),
        query_gpu=query_gpu,
        probe=probe_decode_speed,
    )
    steadyhand_server, steadyhand_preset = steadyhand_fit.server, steadyhand_fit.preset
    os.environ['TOOLSMITH_BASE_URL'] = steadyhand_server.base_url
    STEADYHAND_ACTIVE = {
        'server': steadyhand_server,
        'preset': steadyhand_preset,
        'profile': steadyhand_runtime['profile'],
        'renderer': steadyhand_runtime['renderer'],
        'gpu': steadyhand_fit.gpu,
        'extra_args': steadyhand_fit.extra_args,
        'kv_cache': steadyhand_fit.rung['kv_cache'],
        'flash_attn': steadyhand_fit.rung['flash_attn'],
        'decode_tps': steadyhand_fit.decode_tps,
        'fit_attempts': steadyhand_fit.attempts,
        # The pool admits a little more than parallel so verify or backoff never queues a slot.
        'client': build_llama_client(
            LiveSettings.from_env(api_key=LOCAL_KEY),
            connections=steadyhand_preset.parallel + 2, timeout=900.0,
        ),
    }
    return STEADYHAND_ACTIVE

steadyhand_active = start_steadyhand_runtime()
steadyhand_slot_lines = [line for line in steadyhand_log.read_text(errors='replace').splitlines() if 'n_ctx' in line][:4]
print(json.dumps({
    **steadyhand_active['preset'].as_dict(), 'gpu': steadyhand_active['gpu'], 'extra_args': steadyhand_active['extra_args'],
    'kv_cache': steadyhand_active['kv_cache'], 'flash_attn': steadyhand_active['flash_attn'],
    'self_test_decode_tps': steadyhand_active['decode_tps'], 'attempts': steadyhand_active['fit_attempts'],
}, indent=2))
print('Server log context lines (confirm the per-slot split):', *steadyhand_slot_lines, sep='\n  ')

## Overlap check (part 1)

This cell runs two short scripted episodes at once against the server started above, and records whether their `chat()` calls overlapped in wall-clock time. It starts no model. It reads the server from `TOOLSMITH_BASE_URL`, which the server cell exports, and the model profile and renderer from that cell. If `TOOLSMITH_BASE_URL` is unset or not on localhost, the cell records why it skipped and does nothing else.

Journals go to `/content/steadyhand/<episode_id>.jsonl` on local SSD, with `<episode_id>.manifest.json` beside each one, so Drive FUSE latency stays off the worker. A background copier moves each finished episode to `MyDrive/toolsmith/records/<gpu>/steadyhand/`, and the cell waits for it to drain. Finished journals replay on CPU with `steadyhand.replay.replay_episode`.

In [ ]:
# Part 1 overlap check: two short episodes at once against the running server; records whether
# their model calls overlapped in time. Skips (with a record) if no local server is set.
import concurrent.futures
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

from steadyhand import (
    Budgets,
    LiveConfigError,
    LiveSettings,
    SteadyhandClient,
    build_llama_client,
    build_registry,
    initial_messages,
    load_fixture,
    worker_service,
)
from steadyhand.journal import JournalCopier
from steadyhand.replay import run_claimed_episode
from toolsmith.prompting import ChatPromptRenderer
from toolsmith.protocols import QWEN3_PROFILE, QWEN35_PROFILE
from toolsmith.recording import JsonlRecorder

STEADYHAND_LOCAL_DIR = Path(os.environ.get('STEADYHAND_LOCAL_DIR', '/content/steadyhand'))
STEADYHAND_DRIVE_DIR = Path(DRIVE_RECORDS) / 'steadyhand'
STEADYHAND_SERVER_PARALLEL = int(os.environ.get('TOOLSMITH_PARALLEL', '4'))
STEADYHAND_MAX_STEPS = 4
STEADYHAND_MAX_TOKENS = 512
STEADYHAND_SEED = 0
STEADYHAND_SMOKE_TASKS = (
    ('order-status', 'What is the current status of order ord_1001?'),
    ('payment-check', 'How was order ord_1003 paid for, and has that payment been captured?'),
)
steadyhand_secrets = (HF_TOKEN, LOCAL_KEY)
steadyhand_log = JsonlRecorder(STEADYHAND_DRIVE_DIR / 'part1-smoke.jsonl', secrets=steadyhand_secrets)
steadyhand_profiles = {'qwen3': QWEN3_PROFILE, 'qwen35': QWEN35_PROFILE}
# Use the server cell's profile and renderer when it ran; otherwise fall back to the environment.
steadyhand_active = globals().get('STEADYHAND_ACTIVE') or {}
steadyhand_profile_name = (
    steadyhand_active['profile'].name if steadyhand_active
    else os.environ.get('STEADYHAND_PROFILE', 'qwen3')
)

steadyhand_skip_reason = None
try:
    steadyhand_settings = LiveSettings.from_env(api_key=LOCAL_KEY)
except LiveConfigError as steadyhand_error:
    steadyhand_settings, steadyhand_skip_reason = None, str(steadyhand_error)
if steadyhand_skip_reason is None and steadyhand_profile_name not in steadyhand_profiles:
    steadyhand_skip_reason = f'STEADYHAND_PROFILE must be one of {sorted(steadyhand_profiles)}'

if steadyhand_skip_reason is not None:
    steadyhand_record = steadyhand_log.append({
        'event': 'steadyhand_part1_smoke',
        'status': 'skipped',
        'reason': steadyhand_skip_reason,
    })
    print(f'Steadyhand live smoke skipped: {steadyhand_skip_reason}')
else:
    steadyhand_profile = steadyhand_profiles[steadyhand_profile_name]
    steadyhand_renderer = steadyhand_active.get('renderer')
    if steadyhand_renderer is None:
        steadyhand_renderer = ChatPromptRenderer.from_pretrained(
            steadyhand_profile.tokenizer, token=HF_TOKEN,
        )
    steadyhand_registry = build_registry()
    steadyhand_fixture = load_fixture()
    steadyhand_connections = max(STEADYHAND_SERVER_PARALLEL, len(STEADYHAND_SMOKE_TASKS))
    steadyhand_llama = build_llama_client(
        steadyhand_settings, connections=steadyhand_connections, timeout=600.0,
    )

    class TimedSteadyhandClient(SteadyhandClient):
        """Records perf_counter intervals around each chat() call."""

        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.intervals = []

        def complete(self, messages, tools, *, tool_choice='auto'):
            started = time.perf_counter()
            try:
                return super().complete(messages, tools, tool_choice=tool_choice)
            finally:
                self.intervals.append((started, time.perf_counter()))

    # A fresh label per cell run, so re-running starts new episodes instead of
    # finding the earlier manifests completed and skipping.
    steadyhand_run_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    steadyhand_copier = JournalCopier(STEADYHAND_DRIVE_DIR)

    def run_steadyhand_smoke_episode(label, question):
        # One ToolsmithService per worker; all of them share steadyhand_llama.
        timed_client = TimedSteadyhandClient(
            worker_service(steadyhand_llama, steadyhand_renderer, steadyhand_profile),
            max_tokens=STEADYHAND_MAX_TOKENS,
            seed=STEADYHAND_SEED,
        )
        result = run_claimed_episode(
            STEADYHAND_LOCAL_DIR,
            timed_client,
            steadyhand_registry,
            steadyhand_fixture.clone(),
            initial_messages(question),
            budgets=Budgets(max_steps=STEADYHAND_MAX_STEPS),
            secrets=steadyhand_secrets,
            run_label=f'smoke-{label}-{steadyhand_run_stamp}',
        )
        if result is None:
            return {'label': label, 'status': 'skipped', 'intervals': timed_client.intervals}
        steadyhand_copier.submit(result.journal_path)
        return {
            'label': label,
            'episode_id': result.episode_id,
            'status': result.status.value,
            'steps': result.steps,
            'error_code': result.error_code,
            'final_content': result.final_content,
            'prompt_tokens': result.prompt_tokens,
            'completion_tokens': result.completion_tokens,
            'journal': str(result.journal_path),
            'drive_copy': str(STEADYHAND_DRIVE_DIR / result.journal_path.name),
            'intervals': timed_client.intervals,
        }

    def overlap_seconds(first, second):
        return sum(
            max(0.0, min(first_end, second_end) - max(first_start, second_start))
            for first_start, first_end in first
            for second_start, second_end in second
        )

    steadyhand_started = time.perf_counter()
    try:
        with concurrent.futures.ThreadPoolExecutor(
            max_workers=len(STEADYHAND_SMOKE_TASKS), thread_name_prefix='steadyhand',
        ) as steadyhand_pool:
            steadyhand_futures = [
                steadyhand_pool.submit(run_steadyhand_smoke_episode, label, question)
                for label, question in STEADYHAND_SMOKE_TASKS
            ]
            steadyhand_runs = [future.result() for future in steadyhand_futures]
    finally:
        steadyhand_llama.close()
        steadyhand_copier.close()  # waits until every finished journal is on Drive
    steadyhand_overlap = overlap_seconds(
        steadyhand_runs[0]['intervals'], steadyhand_runs[1]['intervals'],
    )
    for steadyhand_run in steadyhand_runs:
        steadyhand_run['intervals'] = [
            [round(start - steadyhand_started, 4), round(end - steadyhand_started, 4)]
            for start, end in steadyhand_run['intervals']
        ]
    steadyhand_record = steadyhand_log.append({
        'event': 'steadyhand_part1_smoke',
        'status': 'ran',
        'profile': steadyhand_profile.name,
        'model': steadyhand_profile.model,
        'client_connections': steadyhand_connections,
        'episodes': steadyhand_runs,
        'completions_overlapped': steadyhand_overlap > 0,
        'overlap_seconds': round(steadyhand_overlap, 4),
        'copy_errors': steadyhand_copier.errors,
    })
    print(json.dumps(steadyhand_record, indent=2, ensure_ascii=False))

In [ ]:
# Smoke test: one episode alone, then one per slot at once; every slot must be busy. Records
# single-stream and aggregate tokens per second.
import concurrent.futures
import threading
from datetime import datetime, timezone

import httpx

from steadyhand import SteadyhandClient, worker_service
from steadyhand.eval.serving import slots_busy, tokens_per_second
from steadyhand.eval.suite import load_tasks, run_task

if not globals().get('STEADYHAND_ACTIVE'):
    raise RuntimeError('Run the Steadyhand server cell first')
steadyhand_active = STEADYHAND_ACTIVE
steadyhand_parallel = steadyhand_active['preset'].parallel
steadyhand_smoke_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
steadyhand_smoke_dir = Path('/content/steadyhand/smoke') / steadyhand_smoke_stamp
steadyhand_smoke_tasks = [task for task in load_tasks() if task.category == 'single_step'][:steadyhand_parallel]

def steadyhand_live_client(task):
    return SteadyhandClient(
        worker_service(steadyhand_active['client'], steadyhand_active['renderer'], steadyhand_active['profile']),
        max_tokens=int(os.environ.get('STEADYHAND_MAX_TOKENS', '512')),
        seed=task.seed,
        slot_context=steadyhand_active['preset'].per_slot_context,
    )

def steadyhand_smoke_episode(task, label):
    return run_task(
        task, 'clean', policy='typed_hint', directory=steadyhand_smoke_dir / label,
        client=steadyhand_live_client(task), run_label=f'smoke-{label}-{steadyhand_smoke_stamp}',
    )

steadyhand_started = time.perf_counter()
steadyhand_single = steadyhand_smoke_episode(steadyhand_smoke_tasks[0], 'single')
steadyhand_single_tps = tokens_per_second(steadyhand_single.completion_tokens, time.perf_counter() - steadyhand_started)

steadyhand_samples, steadyhand_memory = [], []
steadyhand_sampling = threading.Event()

def steadyhand_sample_slots():
    headers = {'Authorization': f'Bearer {LOCAL_KEY}'}
    with httpx.Client(base_url=steadyhand_active['server'].base_url, headers=headers, timeout=5.0) as http:
        while not steadyhand_sampling.is_set():
            try:
                response = http.get('/slots')
                if response.status_code == 200:
                    steadyhand_samples.append(slots_busy(response.json()))
                steadyhand_memory.append(query_gpu())
            except (httpx.HTTPError, ValueError, subprocess.SubprocessError):
                pass
            time.sleep(0.25)

steadyhand_sampler = threading.Thread(target=steadyhand_sample_slots, daemon=True)
steadyhand_sampler.start()
steadyhand_started = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=steadyhand_parallel) as steadyhand_pool:
    steadyhand_smoke_scores = list(steadyhand_pool.map(
        lambda task: steadyhand_smoke_episode(task, 'aggregate'), steadyhand_smoke_tasks,
    ))
steadyhand_elapsed = time.perf_counter() - steadyhand_started
steadyhand_sampling.set()
steadyhand_sampler.join(timeout=5)

steadyhand_busy = max((busy for busy, _ in steadyhand_samples), default=None)
STEADYHAND_SMOKE = {
    'gpu': steadyhand_active['gpu']['name'],
    'preset': steadyhand_active['preset'].name,
    'parallel': steadyhand_parallel,
    'per_slot_context': steadyhand_active['preset'].per_slot_context,
    'single_stream_tps': steadyhand_single_tps,
    'aggregate_tps': tokens_per_second(sum(s.completion_tokens for s in steadyhand_smoke_scores), steadyhand_elapsed),
    'slots_busy': f'{steadyhand_busy}/{steadyhand_parallel}' if steadyhand_busy is not None else 'unknown',
    'peak_memory_mib': max((row['memory_used_mib'] for row in steadyhand_memory), default=None),
    'peak_gpu_utilization': max((row['utilization_gpu'] for row in steadyhand_memory), default=None),
    'server_alive': steadyhand_active['server'].is_running,
}
print(json.dumps(STEADYHAND_SMOKE, indent=2))
steadyhand_accepted = steadyhand_busy == steadyhand_parallel and STEADYHAND_SMOKE['server_alive']
if not steadyhand_accepted and os.environ.get('STEADYHAND_ACCEPT_SMOKE') != '1':
    raise RuntimeError(
        'Not every slot was busy during the smoke (or /slots is unavailable). Check that the client issues '
        'concurrent complete() calls, or set STEADYHAND_ACCEPT_SMOKE=1 after checking by hand.'
    )

In [ ]:
# The scored sweep: every task, fault variant, policy, and prompt variant, one episode per slot,
# resumable from Drive. Writes RESULTS.md from the scores.
from steadyhand import SteadyhandClient, worker_service
from steadyhand.eval.report import render_report, write_report
from steadyhand.eval.suite import SweepInterrupted, SweepSpec, dataset_sha256, load_tasks, run_sweep
from steadyhand.journal import JournalCopier
from toolsmith.artifacts import copy_file_atomic

if not globals().get('STEADYHAND_ACTIVE') or not globals().get('STEADYHAND_SMOKE'):
    raise RuntimeError('Run the Steadyhand server and smoke cells first')
steadyhand_active = STEADYHAND_ACTIVE
steadyhand_preset = steadyhand_active['preset']
steadyhand_tasks = load_tasks()
steadyhand_policies = tuple(os.environ.get('STEADYHAND_POLICIES', 'typed_hint,escalate').split(','))
steadyhand_prompts = tuple(os.environ.get('STEADYHAND_PROMPTS', 'plain').split(','))
steadyhand_variants = tuple(os.environ.get(
    'STEADYHAND_VARIANTS', 'clean,lost_acknowledgment,explicit_transient,implicit_transient',
).split(','))
steadyhand_max_tokens = int(os.environ.get('STEADYHAND_MAX_TOKENS', '512'))
steadyhand_local = Path('/content/steadyhand')
steadyhand_drive = Path(DRIVE_RECORDS) / 'steadyhand'
steadyhand_spec = SweepSpec(
    sweep_id=f"{steadyhand_preset.profile}-{steadyhand_preset.shape}-{steadyhand_preset.gpu}-"
             f"{'-'.join(steadyhand_policies)}-{'-'.join(steadyhand_prompts)}-{dataset_sha256(steadyhand_tasks)[:8]}",
    task_ids=tuple(task.task_id for task in steadyhand_tasks),
    policies=steadyhand_policies,
    prompt_variants=steadyhand_prompts,
    variants=steadyhand_variants,
    profile=steadyhand_preset.profile,
    dataset_sha256=dataset_sha256(steadyhand_tasks),
    model={
        'model': steadyhand_active['profile'].model, 'use_grammars': True, 'enable_thinking': False,
        'temperature': 0.0, 'max_tokens': steadyhand_max_tokens,
    },
    server={
        **steadyhand_preset.as_dict(), 'gpu': steadyhand_active['gpu']['name'],
        'kv_cache_type': steadyhand_active['kv_cache'], 'flash_attn': steadyhand_active['flash_attn'],
        'batch_size': 2048, 'ubatch_size': 512,
        'extra_args': steadyhand_active['extra_args'],
    },
)

def steadyhand_client_for(unit):
    return SteadyhandClient(
        worker_service(steadyhand_active['client'], steadyhand_active['renderer'], steadyhand_active['profile']),
        max_tokens=steadyhand_max_tokens,
        seed=unit.task.seed,
        slot_context=steadyhand_preset.per_slot_context,
    )

steadyhand_started = time.perf_counter()
ensure_drive()
for steadyhand_attempt in (1, 2):
    try:
        with JournalCopier(steadyhand_drive / 'journals') as steadyhand_copier:
            steadyhand_scores = run_sweep(
                steadyhand_spec, steadyhand_tasks, steadyhand_client_for,
                journal_dir=steadyhand_local / 'journals', sweep_dir=steadyhand_local / 'sweeps',
                workers=steadyhand_preset.parallel, copier=steadyhand_copier,
                sleep=time.sleep, secrets=(HF_TOKEN, LOCAL_KEY),
                # Scores are mirrored to Drive as the sweep runs and restored from it on a fresh runtime,
                # so a disconnect costs only the episodes that were in flight.
                mirror_dir=steadyhand_drive / 'sweeps',
            )
        break
    except SweepInterrupted as steadyhand_error:
        # Episodes kept aborting on infrastructure: the server is down. Restart it once and resume;
        # the aborted episodes are rerun under new episode ids.
        if steadyhand_attempt == 2:
            raise
        print(f'{steadyhand_error}\nRestarting llama-server and resuming the sweep.', flush=True)
        steadyhand_active = start_steadyhand_runtime()
        steadyhand_preset = steadyhand_active['preset']
print(f'Sweep finished in {time.perf_counter() - steadyhand_started:.0f}s; copy errors: {steadyhand_copier.errors}')
steadyhand_report = render_report(steadyhand_scores, sweeps=[steadyhand_spec.as_dict()], smoke=[STEADYHAND_SMOKE])
write_report(steadyhand_local / 'RESULTS.md', steadyhand_report)
copy_file_atomic(steadyhand_local / 'RESULTS.md', steadyhand_drive / 'RESULTS.md')
print(steadyhand_report.split('## 3.')[0])
sync_drive(BACKGROUND, stage='steadyhand-sweep')  # push this cell's results to Google's servers

In [ ]:
# Stop the Steadyhand server (run before switching STEADYHAND_PHASE to another model).
if 'stop_steadyhand_runtime' in globals():
    stop_steadyhand_runtime()
print('Steadyhand llama-server stopped. Start the next phase or model, or disconnect once the Drive copies are done.')

In [ ]:
# Stop the server, let Drive copies finish, stop background jobs, and flush Drive. Keep last.
if 'stop_steadyhand_runtime' in globals():
    stop_steadyhand_runtime()
if 'BACKGROUND' in globals():
    # Let Drive write-backs finish (they are atomic, so an unfinished one is redone next runtime).
    unfinished = BACKGROUND.drain(timeout=float(os.environ.get('TOOLSMITH_CLEANUP_DRAIN_SECONDS', '600')))
    stopped = BACKGROUND.cancel_all()
    if unfinished:
        print(f'Background jobs not finished cleanly: {unfinished}; logs in {BACKGROUND.log_dir}')
    if stopped:
        print(f'Stopped background jobs: {stopped}')
    # Flush every pending Drive write before the runtime goes away (Drive uploads in the background).
    sync_drive(BACKGROUND, stage='cleanup', remount=False)
print('Steadyhand llama-server stopped; Drive flushed and unmounted. Record Colab units used, then delete or disconnect the runtime.')